In [ ]:
# Keep things clean - no annoying warnings
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Importing all requried libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder 
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score,root_mean_squared_error,mean_absolute_error
import os
from tqdm import tqdm

In [ ]:
# Load the glass dataset
df=pd.read_csv("D:\Machine Learning\Datasets\Cases\Glass_Identification\Glass.csv")

# Convert glass types to numbers (sklearn needs numbers, not text)
le=LabelEncoder()
df['Type']=le.fit_transform(df['Type'])

# Split into features(X) and target(y)
X,y=df.drop('Type',axis=1),df['Type']


In [ ]:
# 70-30 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# Time to find the best hyperparameters
# Testing different solvers (optimization methods)

solver=['lbfgs','newton-cg','newton-cholesky','sag','saga']

# C controls regularization - testing 20 values from 0.0001 to 15
Cs=np.linspace(0.0001, 15, 20)

# Try with and without L2 penalty
penalties=['l2', None]

# Store all results here
scores=[]

# Loop through every combination - this might take a while
for s in solver:
    for p in penalties:
        for c in Cs:
            try:
                # Train model with current settings
                lr=LogisticRegression(penalty=p, solver=s, max_iter=1000, C=c)
                lr.fit(X_train, y_train)
                y_pred=lr.predict(X_test)

                # Save the accuracy
                scores.append([s, p, c, accuracy_score(y_test, y_pred)])

            except ValueError as e:
                # Some combinations don't work together - skip them
                print(f"Skipping solver={s}, penalty={p}, C={c:.4f}: {e}")
                scores.append([s, p, c, None])  # Include C value in error case

# Put results in a dataframe for easier viewing            
df_scores=pd.DataFrame(scores, columns=['solver', 'penalty', 'C', 'score'])

# Show me the top 10 performers
print("Top 10 combinations:")
print(df_scores.sort_values('score', ascending=False).head(10))

# Find the best parameters
best_row = df_scores.loc[df_scores['score'].idxmax()]
print(f"\nBest combination:")
print(f"Solver: {best_row['solver']}")
print(f"Penalty: {best_row['penalty']}")
print(f"C: {best_row['C']:.4f}")
print(f"Accuracy: {best_row['score']:.4f}")

Top 10 combinations:
             solver penalty         C     score
50        newton-cg      l2  7.894784  0.707692
47        newton-cg      l2  5.526379  0.707692
49        newton-cg      l2  7.105316  0.707692
46        newton-cg      l2  4.736911  0.707692
48        newton-cg      l2  6.315847  0.707692
89  newton-cholesky      l2  7.105316  0.707692
86  newton-cholesky      l2  4.736911  0.707692
87  newton-cholesky      l2  5.526379  0.707692
88  newton-cholesky      l2  6.315847  0.707692
44        newton-cg      l2  3.157974  0.692308

Best combination:
Solver: newton-cg
Penalty: l2
C: 4.7369
Accuracy: 0.7077


In [ ]:
# Now train the final model with best settings on ALL data
test_model=LogisticRegression(solver='newton-cg',penalty='l2',C=4.7369)
test_model.fit(X,y)

LogisticRegression(C=4.7369, solver='newton-cg')

In [ ]:
# Load test data and make predictions
tst=pd.read_csv("D:\Machine Learning\Datasets\Cases\Glass_Identification\\tst_Glass.csv")
pred_class=test_model.predict(tst)

In [ ]:
# Convert predictions back to original glass type names
tst['predicted']=le.inverse_transform(pred_class)#it is used to get back the label class which we encoded using label encoder

In [ ]:
# Display predicted Results
tst

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,predicted
0,1.5321,14.0,0.00,0.34,70.23,0.001,6.7,1.23,0.000,building_windows_non_float_processed
1,1.5212,15.0,3.00,1.23,75.90,0.100,7.0,0.00,0.440,tableware
2,1.5112,13.0,3.50,2.30,73.00,3.400,14.0,2.30,0.220,building_windows_float_processed
3,1.5000,12.4,1.23,3.22,74.22,4.500,10.0,3.10,0.100,headlamps
4,1.5200,13.0,2.40,0.34,71.22,3.200,9.0,1.44,0.001,building_windows_non_float_processed
5,1.5100,16.0,2.70,4.00,70.00,2.000,6.0,2.90,0.890,headlamps
